# 흥행 영화의 특징은 무엇인가?
## TMDB 데이터 기반 흥행 결정 요인 분석

### 폰트 설정

In [6]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

### 패키지 설치

In [ ]:
%pip install pandas numpy scikit-learn matplotlib seaborn

### import 구성

In [ ]:
# 데이터 처리
import pandas as pd
import numpy as np
import ast
import os

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# ML
from sklearn.preprocessing import StandardScaler # 정규화
from sklearn.decomposition import PCA            # 차원 축소
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

## Step 1. 데이터 전처리

In [ ]:
INPUT_PATH  = r'C:\Users\jinhu\OneDrive\바탕 화면\파이썬 프로젝트\월 데이터마이닝\TMDB_Datamining\data\TMDB_all_movies.csv'
OUTPUT_PATH = r'C:\Users\jinhu\OneDrive\바탕 화면\파이썬 프로젝트\월 데이터마이닝\TMDB_Datamining\data\tmdb_final.csv'

def run_preprocessing(input_path=INPUT_PATH, output_path=OUTPUT_PATH):
    scaled_path = os.path.join(os.path.dirname(output_path), 'tmdb_final_scaled.csv')
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    KEEP_COLS = [
        'budget', 'revenue', 'genres', 'popularity',
        'vote_average', 'vote_count', 'runtime', 'release_date',
        'original_language', 'director', 'cast',
        'imdb_rating', 'imdb_votes', 'status'
    ]

    print("=" * 50)
    print("Step 1: 데이터 로드 및 컬럼 선택")
    print("=" * 50)
    df = pd.read_csv(input_path)
    print(f"원본 데이터: {df.shape[0]:,}행 · {df.shape[1]}컬럼")
    df = df[KEEP_COLS]
    print(f"컬럼 선택 후: {df.shape[0]:,}행 · {df.shape[1]}컬럼")

    print("\n" + "=" * 50)
    print("Step 2: 유효 데이터 필터링")
    print("=" * 50)
    df = df[df['status'] == 'Released']
    print(f"Released 필터링: {df.shape[0]:,}행")
    df = df[(df['budget'] > 10000) & (df['revenue'] > 10000)]
    print(f"budget/revenue 필터링: {df.shape[0]:,}행")
    df = df.drop(columns=['status'])

    print("\n" + "=" * 50)
    print("Step 3: 핵심 변수 결측치 제거")
    print("=" * 50)
    before = len(df)
    df = df.dropna(subset=['genres', 'release_date', 'runtime'])
    print(f"결측치 제거: {before:,} → {len(df):,}행 ({before-len(df):,}개 제거)")

    print("\n" + "=" * 50)
    print("Step 4: 이상치 처리 (IQR × 1.5)")
    print("=" * 50)
    for col in ['budget', 'revenue', 'runtime']:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        before = len(df)
        df = df[(df[col] >= lower) & (df[col] <= upper)]
        print(f"{col}: {before:,} → {len(df):,}행 ({before-len(df):,}개 제거)")
        print(f"  기준: {lower:,.0f} ~ {upper:,.0f}")

    print("\n" + "=" * 50)
    print("Step 5: 날짜 파생 변수 생성")
    print("=" * 50)
    df['release_date'] = pd.to_datetime(df['release_date'])
    df['year']  = df['release_date'].dt.year
    df['month'] = df['release_date'].dt.month
    df['season'] = df['month'].map({
        12: 4, 1: 4, 2: 4,
        3: 1,  4: 1, 5: 1,
        6: 2,  7: 2, 8: 2,
        9: 3, 10: 3, 11: 3
    })
    df = df.drop(columns=['release_date'])
    print("year, month, season 파생 변수 생성 완료")
    print(f"season 분포:\n{df['season'].value_counts().sort_index().rename({1:'봄',2:'여름',3:'가을',4:'겨울'})}")

    print("\n" + "=" * 50)
    print("Step 6: 장르 원-핫 인코딩")
    print("=" * 50)
    def parse_genres(g):
        try:
            items = ast.literal_eval(g)
            if isinstance(items, list):
                return [i['name'] if isinstance(i, dict) else i for i in items]
        except:
            pass
        if isinstance(g, str) and ',' in g:
            return [x.strip() for x in g.split(',')]
        return [g] if isinstance(g, str) and g else []

    df['genres_list'] = df['genres'].apply(parse_genres)
    all_genres = set()
    df['genres_list'].apply(lambda x: all_genres.update(x))
    print(f"전체 장르 수: {len(all_genres)}개")
    print(f"장르 목록: {sorted(all_genres)}")
    for genre in sorted(all_genres):
        if genre:
            col_name = f"genre_{genre.replace(' ', '_').replace('-', '_')}"
            df[col_name] = df['genres_list'].apply(lambda x: 1 if genre in x else 0)
    genre_cols = [c for c in df.columns if c.startswith('genre_')]
    print(f"생성된 장르 컬럼: {len(genre_cols)}개")
    df = df.drop(columns=['genres', 'genres_list'])

    print("\n" + "=" * 50)
    print("Step 7: 감독/출연진 빈도 기반 변수화")
    print("=" * 50)
    df['director'] = df['director'].fillna('Unknown')
    df['cast']     = df['cast'].fillna('Unknown')
    director_counts = df['director'].value_counts()
    TOP_N_DIRECTOR  = 50
    top_directors   = set(director_counts.head(TOP_N_DIRECTOR).index)
    df['director_is_top'] = df['director'].apply(lambda x: 1 if x in top_directors else 0)
    print(f"상위 {TOP_N_DIRECTOR}명 감독 변수화 완료")
    print(f"상위 감독 영화 비율: {df['director_is_top'].mean()*100:.1f}%")

    def get_first_cast(cast_str):
        if pd.isna(cast_str) or cast_str == 'Unknown':
            return 'Unknown'
        try:
            items = ast.literal_eval(cast_str)
            if isinstance(items, list) and len(items) > 0:
                return items[0]['name'] if isinstance(items[0], dict) else items[0]
        except:
            pass
        return cast_str.split(',')[0].strip() if ',' in str(cast_str) else cast_str

    df['lead_actor'] = df['cast'].apply(get_first_cast)
    actor_counts     = df['lead_actor'].value_counts()
    TOP_N_ACTOR      = 50
    top_actors       = set(actor_counts.head(TOP_N_ACTOR).index)
    df['actor_is_top'] = df['lead_actor'].apply(lambda x: 1 if x in top_actors else 0)
    print(f"상위 {TOP_N_ACTOR}명 배우 변수화 완료")
    print(f"상위 배우 영화 비율: {df['actor_is_top'].mean()*100:.1f}%")
    df = df.drop(columns=['director', 'cast', 'lead_actor'])

    print("\n" + "=" * 50)
    print("Step 8: 나머지 결측치 처리")
    print("=" * 50)
    df['imdb_rating'] = df['imdb_rating'].fillna(df['imdb_rating'].median())
    df['imdb_votes']  = df['imdb_votes'].fillna(df['imdb_votes'].median())
    df['original_language'] = df['original_language'].fillna('unknown')
    print(f"결측치 현황:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    print(f"전체 결측치: {df.isnull().sum().sum()}개")

    print("\n" + "=" * 50)
    print("Step 9: 파생 변수 생성")
    print("=" * 50)
    df['roi'] = (df['revenue'] - df['budget']) / df['budget']
    threshold = df['revenue'].quantile(0.70)
    df['hit'] = (df['revenue'] >= threshold).astype(int)
    print(f"ROI 변수 생성 완료 (평균: {df['roi'].mean():.2f})")
    print(f"흥행 기준 revenue: ${threshold:,.0f}")
    print(f"흥행 비율: {df['hit'].mean()*100:.1f}% (상위 30%)")

    print("\n" + "=" * 50)
    print("Step 10: Z-Score 정규화")
    print("=" * 50)
    numeric_cols = [
        'budget', 'popularity', 'vote_average', 'vote_count',
        'runtime', 'imdb_rating', 'imdb_votes', 'year', 'roi'
    ]
    scaler = StandardScaler()
    df_scaled = df.copy()
    df_scaled[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    print(f"정규화 적용 컬럼: {numeric_cols}")

    print("\n" + "=" * 50)
    print("최종 결과")
    print("=" * 50)
    df.to_csv(output_path, index=False)
    df_scaled.to_csv(scaled_path, index=False)
    print(f"최종 데이터셋: {df.shape[0]:,}행 · {df.shape[1]}컬럼")
    print(f"저장 완료: {output_path}")
    print(f"정규화 버전: {scaled_path}")
    print("\n컬럼 목록:")
    for col in df.columns:
        print(f"  - {col}")
    return df, df_scaled

df_pre, df_scaled_pre = run_preprocessing()

## 전처리 결과

**입력:** `TMDB_all_movies.csv`

**출력:**
- `tmdb_final_scaled.csv` 생성 - Z-score 정규화한 값

- `tmdb_final.csv` 생성 

## Step 2. 데이터 로드 및 기본 확인

In [11]:
df = pd.read_csv(r'C:\Users\jinhu\OneDrive\바탕 화면\파이썬 프로젝트\월 데이터마이닝\TMDB_Datamining\data\tmdb_final.csv')
df_scaled = pd.read_csv(r'C:\Users\jinhu\OneDrive\바탕 화면\파이썬 프로젝트\월 데이터마이닝\TMDB_Datamining\data\tmdb_final_scaled.csv')
print(f"데이터셋 크기: {df.shape}")
df.head()

데이터셋 크기: (9278, 35)


,budget,revenue,popularity,vote_average,vote_count,runtime,original_language,imdb_rating,imdb_votes,year,...,genre_Romance,genre_Science_Fiction,genre_TV_Movie,genre_Thriller,genre_War,genre_Western,director_is_top,actor_is_top,roi,hit
0,4000000.0,4257354.0,4.0661,5.900,2821.0,98.0,en,6.7,116918.0,1995,...,0,0,0,0,0,0,0,0,0.064339,0
1,21000000.0,12136938.0,2.1596,6.500,370.0,109.0,en,6.6,21066.0,1993,...,0,0,0,1,0,0,0,0,-0.422051,0
2,839727.0,23218000.0,7.9288,7.977,6018.0,119.0,en,8.2,493614.0,1941,...,0,0,0,0,0,0,0,0,26.649462,1
3,12500000.0,40061153.0,3.4416,7.846,1975.0,140.0,en,7.9,123992.0,2000,...,0,0,0,0,0,0,0,0,2.204892,1
4,5300000.0,1350322.0,5.2261,8.089,3072.0,148.0,de,8.2,201170.0,1927,...,0,1,0,0,0,0,0,0,-0.745222,0


In [12]:
df.describe()

,budget,revenue,popularity,vote_average,vote_count,runtime,imdb_rating,imdb_votes,year,month,...,genre_Romance,genre_Science_Fiction,genre_TV_Movie,genre_Thriller,genre_War,genre_Western,director_is_top,actor_is_top,roi,hit
count,9.278000e+03,9.278000e+03,9278.000000,9278.000000,9278.000000,9278.000000,9278.000000,9.278000e+03,9278.000000,9278.000000,...,9278.000000,9278.000000,9278.000000,9278.000000,9278.000000,9278.000000,9278.000000,9278.000000,9278.000000,9278.000000
mean,1.031631e+07,1.405462e+07,2.784114,6.111389,682.135051,105.278724,6.256564,3.769952e+04,2000.170942,6.831537,...,0.199289,0.076310,0.001832,0.236473,0.034921,0.017353,0.077172,0.019832,3.198065,0.300065
std,1.159472e+07,1.802003e+07,4.229967,1.401626,1287.425369,17.391463,1.077590,8.624572e+04,20.728364,3.448222,...,0.399487,0.265507,0.042768,0.424939,0.183590,0.130589,0.266878,0.139430,20.850740,0.458310
min,1.009400e+04,1.001800e+04,0.000000,0.000000,0.000000,56.000000,1.100000,6.000000e+00,1914.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.999306,0.000000
25%,1.700000e+06,1.215112e+06,1.188425,5.700000,55.000000,93.000000,5.700000,3.269750e+03,1991.000000,4.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.600000,0.000000
50%,6.000000e+06,5.919500e+06,2.134200,6.300000,249.000000,102.000000,6.400000,1.302150e+04,2006.000000,7.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.235507,0.000000
75%,1.500000e+07,2.008437e+07,3.421825,6.900000,773.000000,115.000000,7.000000,3.965300e+04,2015.000000,10.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.849717,1.000000
max,5.900000e+07,7.774597e+07,186.891200,10.000000,30236.000000,155.000000,9.500000,3.185576e+06,2026.000000,12.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,772.060240,1.000000


## Step 3. EDA (탐색적 데이터 분석)



In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.flatten(), ['budget', 'revenue', 'runtime', 'popularity']):
    df[col].hist(ax=ax, bins=50, color='steelblue', edgecolor='white')
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel('빈도')
plt.suptitle('주요 변수 분포', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### 박스플롯

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df.boxplot(column='revenue', ax=axes[0])
axes[0].set_title('revenue 박스플롯')
df.boxplot(column='budget', ax=axes[1])
axes[1].set_title('budget 박스플롯')
plt.tight_layout()
plt.show()

## Step 4. 상관분석

In [ ]:
numeric_cols = ['budget', 'revenue', 'popularity', 'vote_average',
                'vote_count', 'runtime', 'imdb_rating', 'imdb_votes']
corr = df[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5)
plt.title('변수 간 상관관계 히트맵', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
corr_revenue = df[numeric_cols].corr()['revenue'].sort_values(ascending=False)
print("revenue와의 상관계수:")
print(corr_revenue)

## Step 5. PCA (주성분분석)

In [ ]:
pca_cols = ['budget', 'popularity', 'vote_average', 'vote_count',
            'runtime', 'imdb_rating', 'imdb_votes']
X_pca = df_scaled[pca_cols].dropna()

pca = PCA()
pca.fit(X_pca)

explained = pca.explained_variance_ratio_
cumulative = explained.cumsum()

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.bar(range(1, len(explained)+1), explained, color='steelblue')
plt.xlabel('주성분')
plt.ylabel('설명된 분산 비율')
plt.title('Scree Plot')

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumulative)+1), cumulative, 'o-', color='coral')
plt.axhline(y=0.8, color='gray', linestyle='--', label='80%')
plt.xlabel('주성분 수')
plt.ylabel('누적 설명 분산')
plt.title('누적 설명 분산 비율')
plt.legend()
plt.tight_layout()
plt.show()

print("설명된 분산 비율:")
for i, v in enumerate(explained):
    print(f"  PC{i+1}: {v:.4f} ({cumulative[i]:.4f} 누적)")

In [ ]:
pca2 = PCA(n_components=2)
X_2d = pca2.fit_transform(X_pca)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1],
                      c=df.loc[X_pca.index, 'hit'],
                      cmap='coolwarm', alpha=0.5, s=20)
plt.colorbar(scatter, label='흥행 여부 (1=흥행)')
plt.xlabel(f'PC1 ({explained[0]:.1%})')
plt.ylabel(f'PC2 ({explained[1]:.1%})')
plt.title('PCA 2D 시각화 - 흥행/비흥행 분포')
plt.tight_layout()
plt.show()

## Step 6. 모델 학습 및 평가

In [ ]:
feature_cols = ['budget', 'popularity', 'vote_average', 'vote_count',
                'runtime', 'imdb_rating', 'imdb_votes', 'year', 'month',
                'director_is_top', 'actor_is_top']
genre_cols = [c for c in df_scaled.columns if c.startswith('genre_')]
feature_cols = feature_cols + genre_cols

X = df_scaled[feature_cols].dropna()
y = df.loc[X.index, 'revenue']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"학습 데이터: {X_train.shape}")
print(f"테스트 데이터: {X_test.shape}")

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr  = mean_absolute_error(y_test, y_pred_lr)
r2_lr   = r2_score(y_test, y_pred_lr)

print("=== 선형 회귀 성능 ===")
print(f"RMSE: {rmse_lr:,.0f}")
print(f"MAE:  {mae_lr:,.0f}")
print(f"R²:   {r2_lr:.4f}")

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf  = mean_absolute_error(y_test, y_pred_rf)
r2_rf   = r2_score(y_test, y_pred_rf)

print("=== 랜덤 포레스트 성능 ===")
print(f"RMSE: {rmse_rf:,.0f}")
print(f"MAE:  {mae_rf:,.0f}")
print(f"R²:   {r2_rf:.4f}")

In [ ]:
results = pd.DataFrame({
    '모델': ['선형 회귀', '랜덤 포레스트'],
    'RMSE': [rmse_lr, rmse_rf],
    'MAE':  [mae_lr, mae_rf],
    'R²':   [r2_lr, r2_rf]
})
print(results.to_string(index=False))

In [ ]:
importances = pd.Series(rf.feature_importances_, index=feature_cols)
top20 = importances.nlargest(20)

plt.figure(figsize=(10, 7))
top20.sort_values().plot(kind='barh', color='steelblue')
plt.title('랜덤 포레스트 변수 중요도 Top 20')
plt.xlabel('중요도')
plt.tight_layout()
plt.show()

## Step 7. 결론 및 인사이트

### 분석 결과 요약
- **흥행 주요 요인**: (분석 결과 기반으로 작성)
- **모델 성능 비교**: 선형 회귀 vs 랜덤 포레스트
- **제작/투자 전략 인사이트**: (분석 결과 기반으로 작성)